In [15]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q lxml nltk pandas

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
from pathlib import Path
from collections import Counter
import re
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from lxml import html as lxml_html, etree

nltk.download('punkt_tab', quiet=True)
BASE = Path('/content/drive/MyDrive/Text_Mining')
files = sorted((BASE / 'Temp10-Ks').glob('*.txt'))

In [27]:
# Dictionaries
LM = pd.read_csv(BASE / 'Loughran-McDonald_MasterDictionary.csv')
lexicons = {name: set(LM.loc[LM[name] > 0, 'Word'].str.lower())
            for name in ['Positive', 'Negative', 'Litigious']}

with (BASE / 'cmudict.txt').open(encoding='latin-1') as cmu:
    syllableDict = {line.split()[0].lower(): sum(c.isdigit() for c in line)
                    for line in cmu.readlines()[56:] if line.split()}
complex_words = {word for word, count in syllableDict.items() if count > 3}

WORD_RE = re.compile(r"\b[a-z']+\b")
FUTURE_WORDS = {'will', 'shall', 'may', 'might', 'should', 'going', 'gonna'}
FUTURE_RE = re.compile(r"\b(?:will|shall|may|might|should|going|gonna)(?:n't)?\b")
PARSER = lxml_html.HTMLParser(encoding='utf-8', recover=True, huge_tree=True)

def read_filing(path):
    # For SEC submission files, read the first document (the primary filing).
    with path.open('rb') as source:
        header = source.read(4096)
        source.seek(0)
        if b'<SEC-DOCUMENT>' not in header and b'<DOCUMENT>' not in header:
            return source.read()
        chunks, inside = [], False
        for line in source:
            if line.startswith(b'<TEXT>'):
                inside = True
            elif line.startswith(b'</TEXT>') and inside:
                break
            elif inside:
                chunks.append(line)
        return b''.join(chunks)

def clean_text(raw):
    if not raw.strip():
        return ''
    tree = lxml_html.fromstring(raw, parser=PARSER)
    etree.strip_elements(tree, 'script', 'style', with_tail=False)
    return tree.text_content()

def itemseven(text):
    text = text.lower().replace('\xa0', ' ')
    sections = re.findall(
        r"item\s*7\s*[.:–—-]?\s*(management['’]?s?\s*discussion.*?)"
        r"item\s*(?:7\s*a\s*[.:–—-]?\s*quantitative|8\s*[.:–—-]?\s*financial)",
        text, re.S)
    return max(sections, key=len, default='')

def word_stats(counts):
    total = counts.total()
    out = {'Word_Count': total,
           'Total_Complex_Words': sum(counts[w] for w in counts.keys() & complex_words)}
    for name, lexicon in lexicons.items():
        n = sum(counts[w] for w in counts.keys() & lexicon)
        out[name + '_Word_Count'] = n
        rate = {'Positive': 'Positivity', 'Negative': 'Negativity', 'Litigious': 'Litigious'}[name]
        out[rate + '_Rate'] = n / total if total else None
    return out

def fog(words, sentences, complex_count):
    return 0.4 * (words / sentences + 100 * complex_count / words) if words and sentences else None

def analyze(text):
    counts = Counter(WORD_RE.findall(text.lower()))
    sentences = sent_tokenize(text) if text.strip() else []
    out = word_stats(counts)
    out['Total_Sentences'] = len(sentences)
    out['FOG'] = fog(out['Word_Count'], len(sentences), out['Total_Complex_Words'])

    future_counts, future_sentences = Counter(), 0
    for sentence in sentences:
        if FUTURE_RE.search(sentence) and FUTURE_WORDS.intersection(word_tokenize(sentence, preserve_line=True)):
            future_sentences += 1
            future_counts.update(WORD_RE.findall(sentence.lower()))
    future = word_stats(future_counts)
    out.update({
        'Future_Word_Count': sum(future_counts[w] for w in FUTURE_WORDS),
        'Total_Future_Sentences': future_sentences,
        'Future_Complex_Words': future['Total_Complex_Words'],
        'Future_FOG': fog(future['Word_Count'], future_sentences, future['Total_Complex_Words']),
    })
    for key, value in future.items():
        if key not in ('Word_Count', 'Total_Complex_Words'):
            out['Future_' + key] = value
    return out

In [29]:
results = []

for file_path in files:
    text = clean_text(read_filing(file_path))
    mda = itemseven(text)

    row = {'Filename': file_path.name, 'MD&A': mda}
    row.update(analyze(text))
    row.update({'MD&A_' + key: value for key, value in analyze(mda).items()})
    results.append(row)

df = pd.DataFrame(results)

In [32]:
df.to_pickle(BASE / 'Week5_df_final.pkl')
df.head()

,Filename,MD&A,Word_Count,Total_Complex_Words,Positive_Word_Count,Positivity_Rate,Negative_Word_Count,Negativity_Rate,Litigious_Word_Count,Litigious_Rate,...,MD&A_Future_Word_Count,MD&A_Total_Future_Sentences,MD&A_Future_Complex_Words,MD&A_Future_FOG,MD&A_Future_Positive_Word_Count,MD&A_Future_Positivity_Rate,MD&A_Future_Negative_Word_Count,MD&A_Future_Negativity_Rate,MD&A_Future_Litigious_Word_Count,MD&A_Future_Litigious_Rate
0,10-K_1004434_2026-02-17.txt,management’s discussion and analysis of financ...,48251,5093,302,0.006259,686,0.014217,374,0.007751,...,25,22,114,27.470112,2,0.001514,16,0.012112,8,0.006056
1,10-K_1009891_2026-03-27.txt,management’s discussion\nand analysis of finan...,42291,3388,225,0.005320,726,0.017167,468,0.011066,...,24,20,55,16.804838,5,0.007375,8,0.011799,9,0.013274
2,10-K_1018840_2026-03-26.txt,management’s discussion and analysis of financ...,49682,5174,306,0.006159,960,0.019323,535,0.010768,...,88,66,571,35.855316,55,0.010597,221,0.042582,65,0.012524
3,10-K_1023458_2026-03-27.txt,management's discussion and analysis of financ...,14598,1094,38,0.002603,133,0.009111,52,0.003562,...,37,29,76,17.092888,7,0.006836,10,0.009766,3,0.002930
4,10-K_1070494_2026-02-26.txt,management’s discussion and analysis of financ...,78875,8047,609,0.007721,1972,0.025002,1290,0.016355,...,56,45,261,28.030971,21,0.007715,28,0.010287,19,0.006980
